# Example 6.8.3 / 1: Trace a Script with cmdtrace

Companion notebook for **6.8.3 Tracing and Failure Policies** in the AMPL lab Documentation, section "Trace a Script with `cmdtrace`". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Trace a Script with `cmdtrace`

With `option cmdtrace 1;`, AMPL prints each command before it executes it, without pausing as
Stepping Through a Script (AMPL lab documentation, section 6.8.2) does, so a batch log shows the last command reached. The
script below, on the steel model, lowers the available hours, solves and displays the plan.
`ampl.eval()` sends it as one piece of text, as a run file would be read:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PROD;                    # products
param rate {PROD} > 0;       # tons produced per hour
param profit {PROD};         # profit per ton
param market {PROD} >= 0;   # most that can be sold, in tons
param avail >= 0;            # hours available
var Make {p in PROD} >= 0, <= market[p];  # tons produced
maximize Total_Profit: sum {p in PROD} profit[p] * Make[p];
subject to Time: sum {p in PROD} (1 / rate[p]) * Make[p] <= avail;

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["PROD"] = ["bands", "coils"]
ampl.param["rate"] = {"bands": 200, "coils": 140}
ampl.param["profit"] = {"bands": 25, "coils": 30}
ampl.param["market"] = {"bands": 6000, "coils": 4000}
ampl.param["avail"] = 40

In [5]:
# run.py
ampl.eval("""
option cmdtrace 1;
option solver highs, solver_msg 0;
let avail := 35;
solve;
display Make;
option cmdtrace 0;
""")

## 24: option 
## 25: let 
## 26: solve 
## 27: display 
Make [*] :=
bands  6000
coils   700
;

## 28: option 


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
set PROD := bands coils;
param:    rate  profit  market :=
  bands    200      25    6000
  coils    140      30    4000;
param avail := 40;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option cmdtrace 1;
option solver highs, solver_msg 0;
let avail := 35;
solve;
display Make;
option cmdtrace 0;

## 2: option 
## 3: let 
## 4: solve 
## 5: display 
Make [*] :=
bands  6000
coils   700
;

## 6: option 
